# HDBSCAN Overmerge Detection

Run HDBSCAN on each author's work embeddings to detect multi-person profiles.
A clean author should have 1 cluster; an overmerged profile should have 2+.

- Test on 105.3 gold standard (1,122 labeled authors)
- Pull embeddings via authorship_embedding_similarity → work_embeddings_v2
- Job: #105.11 aer-overmerge-signal-calibration

In [ ]:
import numpy as np
import time
from datetime import datetime
from collections import defaultdict
from sklearn.cluster import HDBSCAN

WORK_AUTHORS_TABLE = "openalex.works.work_authors"
WORK_EMBEDDINGS_TABLE = "openalex.vector_search.work_embeddings_v2"
EMBEDDING_DIM = 1024
MIN_WORKS = 5
CHECKPOINT_TABLE = "openalex.aer.hdbscan_overmerge_detection"

## Gold standard author IDs (from job 105.3)

In [ ]:
SPLIT_IDS = {
    5000058354, 5000277470, 5001106162, 5001242576, 5001751485, 5002547846, 5002981530, 5002985815, 5003051525, 5003683328,
    5004699890, 5004799011, 5005483219, 5005776483, 5006626689, 5006832353, 5007114677, 5007831897, 5008757192, 5008769348,
    5008979025, 5009105221, 5009137334, 5009307852, 5010422511, 5010484703, 5010975613, 5011325322, 5011900778, 5012498525,
    5012689737, 5013217186, 5013478230, 5013710631, 5013938260, 5014188192, 5014594712, 5015401837, 5015773218, 5016882470,
    5017142455, 5018155146, 5018218917, 5019217807, 5019862321, 5020336126, 5021173676, 5021967634, 5023003882, 5024177610,
    5025452752, 5027058224, 5027417114, 5027430243, 5028103753, 5029147304, 5029904482, 5030319607, 5030455935, 5031476162,
    5032690462, 5033771456, 5035071631, 5036486793, 5037136506, 5037435179, 5037784680, 5037835036, 5037924045, 5038742031,
    5041864386, 5043303728, 5044626862, 5044690856, 5046337320, 5046389819, 5046438558, 5048151405, 5049568847, 5049695364,
    5050939922, 5051189473, 5051468011, 5052372225, 5052571385, 5052604833, 5053863284, 5054190140, 5054499499, 5054609123,
    5055122935, 5055163904, 5055757812, 5056229359, 5056502482, 5056945568, 5057408903, 5057951893, 5058036156, 5058109631,
    5058421597, 5059488532, 5059871403, 5059901313, 5059938079, 5060030012, 5060512575, 5061138371, 5061782757, 5061844230,
    5062125091, 5062220638, 5062516991, 5063600613, 5063896503, 5064480798, 5064679635, 5065096990, 5065410010, 5065616148,
    5067290858, 5067703574, 5068211137, 5068557265, 5069506861, 5070019932, 5070168570, 5070293380, 5071183569, 5071539405,
    5072305232, 5073218194, 5073480387, 5074059264, 5074251636, 5074303692, 5074432852, 5074654930, 5074948050, 5075292110,
    5075755572, 5076493708, 5076836352, 5078050424, 5079166689, 5079267101, 5079413276, 5079617160, 5079632937, 5079652777,
    5079979698, 5080312909, 5085619921, 5087117526, 5088152507, 5091442005, 5091859986, 5100691568, 5100775309, 5101092520,
    5101114080, 5101441706, 5101536066, 5101597741, 5101672746, 5101690769, 5101833503, 5101839347, 5101989779, 5102111335,
    5102715506, 5102749436, 5102755488, 5103173829, 5103347958, 5103415285, 5103459170, 5103825320, 5103825950, 5103853738,
    5103907138, 5103976114, 5104029031, 5106037625, 5106467052, 5107007469, 5108376978, 5108435508, 5108453412, 5108515912,
    5108570566, 5108613657, 5108623528, 5108748587, 5109066197, 5109085753, 5109148503, 5109293548, 5109296219, 5109831141,
    5110027945, 5110119960, 5110150920, 5110196385, 5110316420, 5110358878, 5110393625, 5110728098, 5110749239, 5110765172,
    5110897558, 5111372795, 5111461672, 5111552276, 5111574283, 5111793979, 5111812935, 5112257425, 5112300394, 5112331881,
    5112390495, 5112522951, 5112655149, 5112702198, 5112723043, 5113171091, 5113370306, 5113385911, 5113434226, 5113550326,
    5113625884, 5113661926, 5113857253, 5113901982, 5113952815, 5113972030, 5114021691, 5114098473, 5114339992, 5117075932,
}

CONFIRMED_IDS = {
    5000075346, 5000163809, 5000185973, 5000294851, 5000296333, 5000304681, 5000447750, 5000469744, 5000470139, 5000484636,
    5000497992, 5000843678, 5000848656, 5000904925, 5000911921, 5001003834, 5001032099, 5001282129, 5001316935, 5001372232,
    5001599295, 5001601670, 5001913247, 5001985433, 5002039026, 5002085848, 5002258179, 5002311944, 5002576950, 5002615359,
    5002926262, 5003078712, 5003172854, 5003420526, 5003483976, 5003505119, 5003686344, 5003746023, 5004184180, 5004367131,
    5004472131, 5004671887, 5004684200, 5004824381, 5004860476, 5004866198, 5004892841, 5005007989, 5005235493, 5005306051,
    5005543226, 5005821958, 5005832450, 5006223967, 5006294349, 5006335520, 5006424579, 5006832353, 5006850062, 5006885020,
    5006891817, 5006892540, 5007072899, 5007433467, 5007658927, 5008411619, 5008653906, 5008753390, 5008811067, 5008932265,
    5009085634, 5009442887, 5009556454, 5009862624, 5010087806, 5010282063, 5010326271, 5010473826, 5010617958, 5010710643,
    5010721126, 5010729263, 5010780031, 5011003163, 5011139306, 5011152330, 5011394544, 5011498947, 5011576899, 5011641801,
    5011644452, 5011679603, 5011693838, 5011916249, 5011946452, 5012021645, 5012168214, 5012245404, 5012319983, 5012343574,
    5012434282, 5012506992, 5012507180, 5012517342, 5012557594, 5012878443, 5013044550, 5013130039, 5013421086, 5013441713,
    5013453399, 5013548603, 5013548657, 5013651353, 5013675121, 5013710631, 5014268293, 5014582331, 5014605473, 5014656544,
    5014820172, 5014825319, 5014857935, 5015279226, 5015297036, 5015418171, 5015678126, 5015768706, 5016174760, 5016361065,
    5017042797, 5017242058, 5017335558, 5017345763, 5017516377, 5017737957, 5017797591, 5018028012, 5018218722, 5018332726,
    5018449545, 5018498851, 5018940029, 5019020069, 5019143490, 5019230935, 5019399805, 5019487908, 5019534047, 5019559449,
    5019689724, 5019694374, 5020073167, 5020298868, 5020336126, 5020686046, 5020779376, 5020779678, 5021118299, 5021417703,
    5021467481, 5021754799, 5022006570, 5022112939, 5022248885, 5022265724, 5022361793, 5022408850, 5022498855, 5022660941,
    5022783704, 5023021809, 5023109586, 5023445015, 5023525193, 5023632964, 5023671070, 5023844756, 5024045056, 5024177610,
    5024520225, 5024736221, 5024885611, 5024913568, 5025452752, 5025594994, 5025673002, 5025676425, 5025716030, 5025733791,
    5026323901, 5026328537, 5026399707, 5026525667, 5026530949, 5026586575, 5026783306, 5026786966, 5026795339, 5026810678,
    5026928687, 5026936966, 5027008194, 5027011498, 5027129242, 5027139411, 5027294842, 5027308708, 5027340715, 5027355126,
    5027442102, 5027464497, 5027618512, 5027652794, 5027673859, 5027746105, 5027805133, 5027823774, 5027967236, 5028121069,
    5028166261, 5028255656, 5028317552, 5028358932, 5028594192, 5028624102, 5028637228, 5028684975, 5028701540, 5029353826,
    5029410157, 5029953650, 5030061430, 5030384394, 5030455749, 5030572484, 5030659236, 5030699621, 5030952362, 5031274503,
    5031422289, 5031446963, 5031468838, 5031901188, 5031943476, 5032027062, 5032154120, 5032442145, 5032562682, 5032594420,
    5032724322, 5032765567, 5032855550, 5032898795, 5032927955, 5033087761, 5033280467, 5033357190, 5033358050, 5033702363,
    5033757301, 5033947013, 5033981216, 5034080774, 5034216548, 5034627185, 5034709306, 5034714440, 5034765161, 5034918628,
    5035315632, 5035366371, 5035379144, 5035452745, 5035470347, 5035638193, 5035806248, 5035844565, 5035999682, 5036015565,
    5036052170, 5036474348, 5036490214, 5036537597, 5036573064, 5036629892, 5036668748, 5036798452, 5036900681, 5036926833,
    5036960709, 5036961392, 5037052967, 5037064192, 5037136506, 5037362549, 5037642196, 5037658628, 5037676122, 5037720618,
    5037834692, 5038366543, 5038391692, 5038456644, 5038573219, 5038629773, 5038725090, 5038771285, 5038881190, 5038936211,
    5039053742, 5039067005, 5039097008, 5039099316, 5039178481, 5039323820, 5039518514, 5039579772, 5039618436, 5039627202,
    5039640098, 5039704204, 5039876954, 5040115708, 5040860738, 5041165489, 5041219736, 5041263823, 5041341396, 5041347112,
    5041511891, 5041625941, 5041727293, 5042250602, 5042268223, 5042423477, 5042617971, 5043160291, 5043275419, 5043532880,
    5043536707, 5043894413, 5043909344, 5044225058, 5044252634, 5044638100, 5044675323, 5044717967, 5044771571, 5045147321,
    5045441478, 5045767147, 5045900441, 5046082715, 5046367806, 5046560031, 5046646750, 5046714003, 5046879461, 5047130741,
    5047273943, 5047405666, 5047889170, 5048020885, 5048092761, 5048345538, 5048434262, 5048717260, 5048840364, 5048893504,
    5048935850, 5049272064, 5049656305, 5049773774, 5049879280, 5050438217, 5050508345, 5050549195, 5050597081, 5050605991,
    5050680576, 5050681681, 5050690992, 5050716540, 5050803435, 5050861768, 5050916310, 5051093942, 5051127099, 5051220527,
    5051240459, 5051394675, 5051475693, 5051765260, 5051971469, 5052438851, 5052446355, 5052577721, 5052625144, 5052638321,
    5053410892, 5053431080, 5053529275, 5053612362, 5053625267, 5053820618, 5053929506, 5053994717, 5054100127, 5054206738,
    5054875075, 5055178909, 5055336708, 5055468124, 5055539335, 5055795702, 5055822075, 5055840391, 5055945041, 5056049276,
    5056097889, 5056104546, 5056196957, 5056381207, 5056448611, 5056564975, 5056784448, 5057122914, 5057243828, 5058038869,
    5058120972, 5058266651, 5058417767, 5058439076, 5058462266, 5058499759, 5058544614, 5058629427, 5058638485, 5058665576,
    5058757303, 5058865617, 5059148355, 5059178961, 5059219746, 5059230576, 5059287105, 5059318810, 5059367962, 5059623594,
    5059995024, 5060426419, 5060447715, 5060492110, 5060502849, 5060557374, 5060634385, 5060679267, 5060692667, 5061156571,
    5061312215, 5061313357, 5061317367, 5061346706, 5061377970, 5061672657, 5061800499, 5062005589, 5062149527, 5062318322,
    5062324748, 5062402898, 5062443593, 5062915144, 5063093897, 5063197873, 5063221355, 5063427196, 5063437398, 5063543149,
    5063577042, 5063634793, 5063641432, 5063677120, 5063684507, 5063722751, 5063820039, 5063882163, 5063896503, 5064292242,
    5064394052, 5064577083, 5064662238, 5064819693, 5065274638, 5065305947, 5065396267, 5065673941, 5065782376, 5065930587,
    5066012580, 5066075206, 5066124580, 5066242985, 5066300307, 5066311192, 5066462242, 5066504722, 5066656489, 5066802249,
    5066897101, 5066898814, 5067349056, 5067354901, 5067395854, 5067818038, 5067944927, 5068126897, 5068545785, 5068611865,
    5068645545, 5068725063, 5069131323, 5069134214, 5069451727, 5069503481, 5069711321, 5069820712, 5070049195, 5070087165,
    5070168570, 5070456575, 5070641856, 5070662386, 5070669440, 5070734918, 5071015651, 5071117027, 5071145958, 5071489338,
    5071687822, 5071827299, 5071831846, 5072093741, 5072137981, 5072146909, 5072242410, 5072344503, 5072354102, 5072454436,
    5072647734, 5072700564, 5072943654, 5073013669, 5073053884, 5073314287, 5073380969, 5073506236, 5073535794, 5073924686,
    5074088812, 5074095425, 5074343057, 5074606990, 5074731779, 5074749021, 5074839838, 5074953319, 5075014758, 5075056645,
    5075332736, 5075344794, 5075369944, 5075560544, 5075570397, 5075773797, 5075946256, 5076274599, 5076376736, 5076772435,
    5076871288, 5077040304, 5077074056, 5077081952, 5077474991, 5077517855, 5077560797, 5077799739, 5077815186, 5077918810,
    5077919213, 5078138160, 5078493275, 5078554832, 5078617631, 5078623160, 5078739501, 5078773743, 5078821164, 5079094137,
    5079105244, 5079259682, 5079299797, 5079485983, 5079497159, 5079654941, 5079798553, 5080161180, 5080223176, 5080233397,
    5080377930, 5080492356, 5080541937, 5080591937, 5080849431, 5080936679, 5080974491, 5081020524, 5081259536, 5081747147,
    5082325782, 5082406391, 5083252906, 5083501260, 5083508133, 5083515378, 5083631981, 5084195402, 5084370640, 5084506603,
    5084529942, 5084549388, 5084897975, 5085326237, 5085714281, 5086191294, 5087090962, 5087110857, 5087166684, 5087284111,
    5087351364, 5087627130, 5088088732, 5088150711, 5088159296, 5088382884, 5088414341, 5088642071, 5088673726, 5088868136,
    5088931673, 5089180563, 5089585472, 5089638676, 5089935956, 5089987004, 5090037794, 5090576103, 5090821911, 5091169973,
    5091351524, 5091444955, 5093539865, 5093794367, 5099778923, 5100460107, 5100749128, 5101218631, 5101499542, 5101570513,
    5101621719, 5101711853, 5101779418, 5101781341, 5101784338, 5101994511, 5101995501, 5102295455, 5102752646, 5102849637,
    5102850112, 5102983380, 5103010733, 5103017316, 5103142430, 5103242520, 5103243948, 5103270389, 5103478714, 5103853738,
    5103920521, 5104050138, 5104064622, 5104353842, 5105090549, 5106499892, 5107864039, 5108260432, 5108303598, 5108505492,
    5108603487, 5108606319, 5109059349, 5109124248, 5109147998, 5109164611, 5109201858, 5109334300, 5109546853, 5109604058,
    5109782520, 5109913855, 5109968665, 5109989246, 5109991333, 5109993381, 5110026802, 5110183202, 5110246466, 5110331792,
    5110381357, 5110570821, 5111001074, 5111681297, 5111689101, 5111838429, 5111945307, 5111962545, 5112050864, 5112062575,
    5112153887, 5112171421, 5112374216, 5112498225, 5112782858, 5112789394, 5112811186, 5113278113, 5113547932, 5113549983,
    5113623662, 5113699011, 5113717938, 5113784033, 5113877625, 5114030225, 5114631031, 5120882986, 5120901273, 5123677405,
    5126137456, 5126304874, 5129641886,
}

# Resolve overlaps: if in both, label as split (conservative)
overlap = SPLIT_IDS & CONFIRMED_IDS
CONFIRMED_IDS = CONFIRMED_IDS - overlap
ALL_IDS = SPLIT_IDS | CONFIRMED_IDS

print(f'Split: {len(SPLIT_IDS)}, Confirmed: {len(CONFIRMED_IDS)}, Overlap removed: {len(overlap)}, Total: {len(ALL_IDS)}')

## Process in batches: pull embeddings + run HDBSCAN + checkpoint per batch

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DoubleType
from sklearn.metrics import silhouette_score

STAGED_TABLE = "openalex.aer.temp_hdbscan_work_ids"
BATCH_SIZE = 100  # authors per batch

RESULT_SCHEMA = StructType([
    StructField('author_id', LongType()),
    StructField('label', StringType()),
    StructField('work_count', IntegerType()),
    StructField('n_clusters', IntegerType()),
    StructField('noise_frac', DoubleType()),
    StructField('silhouette', DoubleType()),
    StructField('largest_cluster_frac', DoubleType()),
    StructField('second_cluster_frac', DoubleType()),
])

# Setup checkpoint table and find already-done authors
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHECKPOINT_TABLE} (
        author_id BIGINT, label STRING, work_count INT,
        n_clusters INT, noise_frac DOUBLE, silhouette DOUBLE,
        largest_cluster_frac DOUBLE, second_cluster_frac DOUBLE
    )
""")
done_ids = set(r['author_id'] for r in spark.sql(f"SELECT author_id FROM {CHECKPOINT_TABLE}").collect())

# Build todo list
todo = sorted([a for a in ALL_IDS if a not in done_ids])
total = len(todo)
print(f"TODO: {total} authors ({len(done_ids)} already done)")
print("=" * 90)

errors = []
start_time = time.time()
processed = 0

for batch_start in range(0, total, BATCH_SIZE):
    batch_authors = todo[batch_start:batch_start + BATCH_SIZE]
    batch_ids_str = ','.join(str(a) for a in batch_authors)
    batch_t0 = time.time()

    # Pull embeddings for this batch only
    rows = spark.sql(f"""
        SELECT t.author_id, e.work_id, e.embedding
        FROM {STAGED_TABLE} t
        JOIN {WORK_EMBEDDINGS_TABLE} e ON t.work_id = e.work_id
        WHERE t.author_id IN ({batch_ids_str})
          AND e.embedding IS NOT NULL
    """).collect()

    # Group by author
    from collections import defaultdict
    author_embs = defaultdict(list)
    for row in rows:
        emb = np.array(row['embedding'][:EMBEDDING_DIM], dtype=np.float32)
        if not np.any(np.isnan(emb)):
            author_embs[row['author_id']].append(emb)

    # Run HDBSCAN per author
    batch_results = []
    for author_id in batch_authors:
        label = 'split' if author_id in SPLIT_IDS else 'confirmed'
        embeddings = author_embs.get(author_id, [])
        n_works = len(embeddings)

        if n_works < MIN_WORKS:
            continue

        try:
            X = np.array(embeddings)
            clusterer = HDBSCAN(min_cluster_size=3, min_samples=2, metric='cosine', algorithm='brute')
            labels = clusterer.fit_predict(X)

            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            noise_frac = float(np.sum(labels == -1)) / len(labels)

            non_noise = labels != -1
            if n_clusters >= 2 and np.sum(non_noise) >= 2:
                sil = float(silhouette_score(X[non_noise], labels[non_noise], metric='cosine'))
            else:
                sil = 0.0

            if n_clusters > 0:
                cluster_ids = [c for c in set(labels) if c != -1]
                sizes = sorted([int(np.sum(labels == c)) for c in cluster_ids], reverse=True)
                largest_frac = float(sizes[0]) / n_works
                second_frac = float(sizes[1]) / n_works if len(sizes) > 1 else 0.0
            else:
                largest_frac = 0.0
                second_frac = 0.0

            batch_results.append((author_id, label, n_works, n_clusters, noise_frac,
                                  sil, largest_frac, second_frac))
        except Exception as e:
            errors.append({'author_id': author_id, 'error': str(e)})

    # Checkpoint this batch
    if batch_results:
        spark.createDataFrame(batch_results, schema=RESULT_SCHEMA).write.mode("append").saveAsTable(CHECKPOINT_TABLE)

    processed += len(batch_authors)
    elapsed = time.time() - start_time
    rate = processed / elapsed if elapsed > 0 else 0
    eta = (total - processed) / rate if rate > 0 else 0
    batch_elapsed = time.time() - batch_t0
    multi = sum(1 for r in batch_results if r[3] >= 2)
    print(f"  [{processed}/{total}] batch {batch_elapsed:.1f}s | {len(batch_results)} results ({multi} multi-cluster) | "
          f"{elapsed:.0f}s elapsed, ETA {eta:.0f}s | {len(errors)} errors")

total_done = spark.sql(f"SELECT COUNT(*) AS n FROM {CHECKPOINT_TABLE}").collect()[0]['n']
print("=" * 90)
print(f"Done: {total_done} authors in checkpoint table, {len(errors)} errors, {time.time()-start_time:.0f}s")

## Summary

In [ ]:
rows = spark.sql(f'SELECT * FROM {CHECKPOINT_TABLE}').collect()
results_dict = [r.asDict() for r in rows]

output_lines = []
output_lines.append(f'Authors analyzed: {len(results_dict)}')
output_lines.append('')

# Key question: does n_clusters >= 2 predict overmerge?
for label in ['split', 'confirmed']:
    subset = [r for r in results_dict if r['label'] == label]
    multi = sum(1 for r in subset if r['n_clusters'] >= 2)
    output_lines.append(f'{label} (n={len(subset)}): {multi} have 2+ clusters ({multi/len(subset)*100:.0f}%)')

output_lines.append('')
output_lines.append('METRIC COMPARISON (median split vs confirmed):')
output_lines.append('=' * 80)

metrics = ['n_clusters', 'noise_frac', 'silhouette', 'largest_cluster_frac', 'second_cluster_frac']
for metric in metrics:
    split_vals = [r[metric] for r in results_dict if r['label'] == 'split' and r[metric] is not None]
    conf_vals = [r[metric] for r in results_dict if r['label'] == 'confirmed' and r[metric] is not None]
    if split_vals and conf_vals:
        s_med = np.median(split_vals)
        c_med = np.median(conf_vals)
        gap = abs(s_med - c_med)
        pooled_std = np.std(split_vals + conf_vals)
        effect = gap / pooled_std if pooled_std > 0 else 0
        direction = 'higher' if s_med > c_med else 'lower'
        output_lines.append(f'  {metric:25s}: split={s_med:.4f} confirmed={c_med:.4f} gap={gap:.4f} effect={effect:.2f} ({direction})')

# Classification accuracy at n_clusters >= 2 threshold
output_lines.append('')
output_lines.append('CLASSIFICATION (n_clusters >= 2 = predict overmerge):')
tp = sum(1 for r in results_dict if r['label'] == 'split' and r['n_clusters'] >= 2)
fp = sum(1 for r in results_dict if r['label'] == 'confirmed' and r['n_clusters'] >= 2)
fn = sum(1 for r in results_dict if r['label'] == 'split' and r['n_clusters'] < 2)
tn = sum(1 for r in results_dict if r['label'] == 'confirmed' and r['n_clusters'] < 2)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
output_lines.append(f'  TP={tp} FP={fp} FN={fn} TN={tn}')
output_lines.append(f'  Precision={precision:.2f} Recall={recall:.2f} F1={f1:.2f}')

if errors:
    output_lines.append(f'\nERRORS ({len(errors)}):')
    for e in errors[:10]:
        output_lines.append(f'  A{e["author_id"]}: {e["error"]}')

output = '\n'.join(output_lines)
print(output)
dbutils.notebook.exit(output)